# System Design

## *Nina Haide*

## Introduction

### Industry Context

The selected industry is **AI talent recruitment and career intelligence**,
specifically the European market for AI professionals.
This is a rapidly growing sector characterised by a structural imbalance
between employer demand for AI skills and the availability of qualified
candidates. In 2024, 13.5% of EU enterprises used AI, up from just 8% in
2023,  accelerating demand for AI talent across all sectors and
making career intelligence tools increasingly relevant for both candidates
and employers.

Existing platforms such as Jobright.ai demonstrate strong product-market
fit for AI-assisted job matching in the United States, offering personalised
job matches, one-click application autofill, tailored resume generation,
insider referral connections and a 24/7 AI career copilot to over 1.25
million users. However, Jobright.ai operates primarily in the US market
and does not serve the European context with its distinct regulatory
environment, multilingual job postings, salary structures and
fragmented data landscape.

On the employer-facing side, platforms such as Beamery, Eightfold AI
and Phenom - often described as talent intelligence platforms - use
advanced machine learning to infer skills from résumés and recommend
candidates for open roles.  However, these are B2B tools designed
for recruiters rather than candidates, operate primarily at enterprise
scale and do not provide the kind of transparent, candidate-facing
career guidance that this project targets. LinkedIn launched an AI-powered
Hiring Assistant in 2024 that can draft job descriptions, automatically
source candidates and initiate outreach - but LinkedIn explicitly
prohibits automated scraping of its platform in its Terms of Service,
making it unavailable as a data source for academic or independent
research. The same restriction applies to Indeed, Glassdoor and most
other major job platforms, which either prohibit scraping entirely or
gate their data behind expensive enterprise APIs. This makes the Adzuna
Jobs API the most viable open data source for European job market
research.

The primary data source for this project is the **Adzuna Jobs API**.
Adzuna is a British company founded in London in 2011 that now has
offices in South West London, Minneapolis, Paris and Sydney, operating
in 19 countries across Europe, North America and Australia. It
aggregates job listings from employer career pages, recruitment agencies
and job boards into a single searchable index and supplies real-time
data to the UK's Number 10 Dashboard and Office for National Statistics
labour market indices.  Unlike US-centric platforms, Adzuna's European
origins and coverage across Germany, Austria, Switzerland, France, the
Netherlands and the UK make it the most suitable open data source for
building a European-focused career intelligence system.

This project designs and partially implements a **European AI Career
Intelligence System (EACIS)** - a coherent, data-driven platform
combining labour market analytics, semantic role understanding and
agentic career guidance into a single integrated system. It is
positioned as a European-focused, transparency-first, candidate-facing
alternative to the employer-facing talent intelligence tools currently
dominating the European market, with EU AI Act compliance built into
the architecture from the outset.

### Problem Definition

The core problem is a **three-sided information asymmetry** in the
European AI job market:

1. **Candidates** do not know which AI roles their skills qualify them
   for, what those roles pay or how to bridge the gap between their
   current profile and their target role
2. **Employers** struggle to define role requirements consistently -
   job titles are unreliable proxies for actual skill demand, leading
   to mismatched applications and inefficient hiring
3. **The market** lacks a structured, data-driven taxonomy of AI roles
   specific to the European context — existing classifications are
   either US-derived or too generic to reflect the skill profiles
   actually in demand

### AI Suitability Assessment

This problem is well-suited to AI-based solutions for three reasons.
First, the scale of the data - hundreds of thousands of job postings
and candidate profiles - exceeds human capacity to analyse manually.
Second, the signal is latent: the meaningful structure of the job
market (skill clusters, salary patterns, role archetypes) is not
explicitly labelled anywhere and must be learned from data. Third,
the guidance task - mapping a candidate's profile to a market archetype
and generating a personalised recommendation - requires the kind of
contextual language understanding that modern NLP models are
specifically designed to provide.

### Constraints, Risks and Ethical Considerations

**Regulatory constraints:** The EU AI Act officially entered into
force in August 2024, with core requirements for high-risk AI systems
including documentation, human oversight and audits becoming
enforceable in August 2026.  Recruitment and HR systems are classified
as high-risk under Annex III, requiring conformity assessment and bias
monitoring before deployment - a constraint that Jobright.ai, operating
under US regulation, does not currently face but that EACIS treats as
a design requirement rather than a compliance burden.

**Data constraints:** The Adzuna API truncates job descriptions at
approximately 500 characters and 18% of postings could not be enriched
via secondary fetching due to JavaScript-rendered employer sites.
Geographic imbalance - Germany and the UK account for over 80% of
collected postings - means the system's archetypes primarily reflect
German and British AI hiring norms rather than the full European market.

**Competitive constraints:** Established talent intelligence platforms
and LinkedIn have significant advantages in data volume, engineering
resources and network effects. A European-focused system would need
to differentiate on regulatory compliance, transparency and
candidate-centric design - areas where the current prototype
demonstrates genuine architectural advantages over black-box commercial
systems.

**Ethical risks:** The primary misuse risk is repurposing the
candidate-facing guidance system as a recruiter-facing screening tool.
Archetype similarity scores could be used to filter or rank applicants,
encoding the biases present in the training data into hiring decisions.
A second ethical risk is the system's confident-sounding recommendations
without uncertainty quantification - candidates may act on a no-fit
response without realising their competencies may fall outside the
narrow 50-skill taxonomy.

## Integration Plan

### System Overview

The European AI Career Intelligence System (EACIS) integrates all six
prior capstone projects into a unified pipeline. Each project contributes
a distinct layer of the system - from raw data collection through
statistical analysis, unsupervised role discovery, semantic representation
learning, generative guidance and autonomous orchestration. Each layer depends on the
outputs of the previous one and the final agentic system synthesises
all components into a single end-to-end workflow.

### Project Contributions

**Project 1 - Data Collection**
*Contribution: Data infrastructure layer*

The Adzuna API collection scripts (`fetch_ai_jobs_and_skills.py` for
DACH, `fetch_ai_jobs_personas.py` for the extended European scope) form
the data foundation of the entire system. They collect structured job
postings across six European countries using targeted search queries
spanning five role categories, and the description enrichment script
(`enrich_descriptions.py`) recovers full job description text by
fetching source pages directly - overcoming the API's 500-character
truncation limit. The pipeline also establishes the geographic scope, the role
category taxonomy and the data quality standards - including outlier
handling and description length filtering.

**Project 2 — Statistical Salary Analysis**
*Contribution: Market intelligence layer*

The DACH salary analysis provides the compensation context embedded in
every career report generated by the system. EUR-normalised salary
statistics by archetype — derived from the ai-jobs.net dataset using
live ECB exchange rates via the Frankfurter API — give candidates
concrete salary expectations rather than generic guidance. The
hypothesis test confirming that remote work mode significantly affects
salary (F=4.56, p=0.0107) also informs the salary range estimates,
which are segmented by archetype and work mode rather than generic
market averages. The IDA framework applied in Project 2 established
the data quality standards — outlier removal, currency normalisation and
missing value handling — that were carried forward as a methodological
baseline into all subsequent projects.

**Project 3 — Skill Extraction and Role Clustering**
*Contribution: Skill taxonomy and role archetypes*

The 50-skill taxonomy developed in Project 3 is the extraction engine
used throughout the entire system — by the data enrichment pipeline,
the resume preprocessing step in Project 5 and the skill extraction
tool in the agentic agent. The k-means clustering of DACH job postings
by binary skill vectors produced the five foundational role archetypes
(Data & Analytics Generalist, Cloud ML Engineer, MLOps & GenAI
Engineer, AI Automation & Integration, Deep Learning & AI Research)
that define the matching targets for all downstream components. The
silhouette score evaluation and cluster cross-tabulation established
that AI_Engineering dominates three of five clusters — a finding that
directly motivated the geographic expansion to the UK and Europe in
Project 4 to obtain a more balanced dataset.

**Project 4 — Deep Learning Persona Discovery**
*Contribution: Semantic representation layer and FAISS index*

Project 4 replaced keyword-based skill vectors with DistilBERT semantic
embeddings, encoding the full meaning of each job description into a
768-dimensional vector. This produced a richer and more nuanced set of
European AI job market personas, and the experimental comparison of
TF-IDF versus DistilBERT clustering provided empirical justification
for using Transformer embeddings as the production representation in
EACIS - the DistilBERT model produced more coherent and semantically
meaningful clusters than the TF-IDF baseline. The FAISS index of 1,088
DistilBERT-encoded job descriptions from this project is the retrieval
backbone of the RAG component in the agentic system, enabling semantic
similarity search at query time without requiring re-embedding.

**Project 5 — Generative AI and Recommendation**
*Contribution: Candidate-facing language generation*

Project 5 established the recommendation generation pipeline — fine-tuning
GPT-2 on structured prompt-response pairs derived from 2,484 real
resumes sourced from livecareer.com. While the final agentic system
uses deterministic templates rather than live GPT-2 generation due to
hallucinations observed in testing, the fine-tuning experiment from
Project 5 directly informed this architectural decision. The three-tier
fit routing (strong / partial / none), the structured prompt template
format, the cosine similarity thresholds and the post-processing
pipeline were all developed and validated in Project 5 before being
integrated into the agent. The GPT-2 experiment also provided empirical
evidence that small language models require either larger fine-tuning
datasets or stronger base models to produce reliable domain-specific
text - a finding that informs the future improvements roadmap.

**Project 6 — Agentic AI Orchestration**
*Contribution: End-to-end autonomous workflow*

Project 6 synthesises all prior components into a single ReAct agent
that orchestrates the full career guidance workflow autonomously. The
agent uses the Project 3 taxonomy for skill extraction, the Project 3
centroids for archetype matching, the Project 4 FAISS index for RAG
retrieval, the Project 5 recommendation templates for guidance
generation, the Project 1 Adzuna API for live job fetching and the
Project 2 salary statistics for compensation context. The state
dictionary and reasoning trace provide session memory and full
transparency, addressing the explainability requirements relevant to
EU AI Act compliance.


### Component Interaction Map

### Integration Dependencies

The components form a strict dependency chain — each layer requires
the outputs of all previous layers to function. The skill taxonomy
(P3) must exist before skill extraction can run. The archetypes (P3)
and FAISS index (P4) must exist before the agent can match or retrieve.
The recommendation templates (P5) must be defined before the agent can
generate guidance. The salary lookup (P2) must be pre-computed before
the agent can include compensation context. The Adzuna pipeline (P1)
must collect and enrich data before any modelling is possible.

This dependency chain means the system cannot be partially deployed -
all components must be present and consistent for the end-to-end
workflow to execute correctly. It also means that if the underlying
job market changes significantly - new roles emerge, skill terminology
shifts - the entire pipeline from P1 through P6 must be retrained and
rebuilt from fresh data. This is a known architectural risk documented
explicitly in the synthesis paper.